In [ ]:
pip install yfinance pandas numpy matplotlib

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
#  GOOFY SCREENER — Phase 2
#  Input any list of assets → outputs which strategy fits each one best
#  Strategies: MA Crossover | RSI | Bollinger Bands | MACD | Mean Reversion
# ══════════════════════════════════════════════════════════════════════════════

# ── CHANGE THIS LIST TO SCREEN ANY ASSETS YOU WANT ────────────────────────────
ASSETS = ["NVDA", "SPY", "CBA.AX", "BHP.AX", "7203.T", "6758.T"]

# ── Date range (same train/test split as all 5 strategy notebooks) ─────────────
TRAIN_START = dt.datetime(2016, 1, 1)
TRAIN_END   = dt.datetime(2021, 1, 1)
TEST_END    = dt.datetime.now()

PERIODS_PER_YEAR = 252

print("GOOFY SCREENER — Phase 2")
print(f"Assets to screen: {ASSETS}")
print(f"Train: {TRAIN_START.date()} → {TRAIN_END.date()}")
print(f"Test:  {TRAIN_END.date()} → {TEST_END.date()}")
print("\nDownloading price data...")

price_data = {}
for asset in ASSETS:
    raw = yf.download(asset, start=TRAIN_START, end=TEST_END,
                      auto_adjust=True, progress=False)
    if raw.empty:
        print(f"  WARNING: No data for {asset} — skipping")
    else:
        price_data[asset] = raw["Close"].squeeze()
        print(f"  {asset}: {len(raw)} rows")

ASSETS = list(price_data.keys())  # remove any that failed to download
print(f"\nReady to screen {len(ASSETS)} assets.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STRATEGY DEFINITIONS
#  Each strategy returns: position series (0 or 1), one row per day
#  All use the same log-return / Sharpe / drawdown calculation at the end
# ══════════════════════════════════════════════════════════════════════════════

# ── Shared metrics helper ──────────────────────────────────────────────────────
def compute_metrics(price_series, position_series):
    """Given a price series and a 0/1 position series, return key metrics."""
    df = pd.DataFrame({"price": price_series})
    df["position"]            = position_series.reindex(df.index).fillna(0)
    df["log_ret"]             = np.log(df["price"] / df["price"].shift(1))
    df["strat_ret"]           = df["position"] * df["log_ret"]
    df["cum_market"]          = np.exp(df["log_ret"].cumsum())
    df["cum_strategy"]        = np.exp(df["strat_ret"].cumsum())

    strat_lr  = df["strat_ret"].dropna()
    ann_ret   = np.exp(strat_lr.mean() * PERIODS_PER_YEAR) - 1
    vol       = strat_lr.std() * np.sqrt(PERIODS_PER_YEAR)
    sharpe    = ann_ret / vol if vol != 0 else np.nan
    total_ret = df["cum_strategy"].dropna().iloc[-1] - 1 if len(df["cum_strategy"].dropna()) > 0 else np.nan

    cum = df["cum_strategy"].dropna()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min() if len(cum) > 0 else np.nan

    return {
        "Sharpe":       round(sharpe,    3),
        "Total Ret %":  round(total_ret * 100, 2),
        "Ann Ret %":    round(ann_ret   * 100, 2),
        "Max DD %":     round(mdd       * 100, 2),
        "_df":          df,
    }


# ── 1. Moving Average Crossover ────────────────────────────────────────────────
# Buy when short MA crosses above long MA (uptrend starting)
# Exit when short MA crosses below long MA (trend reversing)
def strategy_ma(price, short=20, long=50):
    ma_short = price.rolling(short).mean()
    ma_long  = price.rolling(long).mean()
    signal   = (ma_short > ma_long).astype(int).shift(1)
    return signal


# ── 2. RSI ─────────────────────────────────────────────────────────────────────
# Buy when RSI drops below oversold threshold (selling exhausted)
# Exit when RSI rises above overbought threshold (buying exhausted)
def strategy_rsi(price, period=14, oversold=30, overbought=70):
    delta   = price.diff()
    gain    = delta.clip(lower=0).rolling(period).mean()
    loss    = (-delta.clip(upper=0)).rolling(period).mean()
    rs      = gain / loss
    rsi     = 100 - (100 / (1 + rs))
    signal  = np.zeros(len(price))
    in_trade = False
    for i in range(len(rsi)):
        r = rsi.iloc[i]
        if np.isnan(r):
            signal[i] = 0
        elif not in_trade and r < oversold:
            in_trade  = True
            signal[i] = 1
        elif in_trade and r > overbought:
            in_trade  = False
            signal[i] = 0
        else:
            signal[i] = 1 if in_trade else 0
    return pd.Series(signal, index=price.index).shift(1)


# ── 3. Bollinger Bands ─────────────────────────────────────────────────────────
# Buy when price touches the lower band (statistically cheap)
# Exit when price reverts to the middle band (mean)
def strategy_bb(price, window=20, num_std=2.0):
    mid    = price.rolling(window).mean()
    std    = price.rolling(window).std()
    lower  = mid - num_std * std
    signal = np.zeros(len(price))
    in_trade = False
    for i in range(len(price)):
        p = price.iloc[i]
        m = mid.iloc[i]
        l = lower.iloc[i]
        if np.isnan(l):
            signal[i] = 0
        elif not in_trade and p <= l:
            in_trade  = True
            signal[i] = 1
        elif in_trade and p >= m:
            in_trade  = False
            signal[i] = 0
        else:
            signal[i] = 1 if in_trade else 0
    return pd.Series(signal, index=price.index).shift(1)


# ── 4. MACD ────────────────────────────────────────────────────────────────────
# Buy when MACD line crosses above signal line (short-term momentum accelerating)
# Exit when MACD line crosses below signal line (momentum fading)
def strategy_macd(price, fast=12, slow=26, signal_period=9):
    ema_fast    = price.ewm(span=fast,          adjust=False).mean()
    ema_slow    = price.ewm(span=slow,          adjust=False).mean()
    macd_line   = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal_period, adjust=False).mean()
    signal      = (macd_line > signal_line).astype(int).shift(1)
    return signal


# ── 5. Mean Reversion (Z-score) ────────────────────────────────────────────────
# Buy when price is unusually far below its rolling mean (Z-score < -threshold)
# Exit when price reverts back to the mean (Z-score > 0)
def strategy_mr(price, window=20, threshold=1.5):
    roll_mean = price.rolling(window).mean()
    roll_std  = price.rolling(window).std()
    zscore    = (price - roll_mean) / roll_std
    signal    = np.zeros(len(price))
    in_trade  = False
    for i in range(len(zscore)):
        z = zscore.iloc[i]
        if np.isnan(z):
            signal[i] = 0
        elif not in_trade and z < -threshold:
            in_trade  = True
            signal[i] = 1
        elif in_trade and z >= 0:
            in_trade  = False
            signal[i] = 0
        else:
            signal[i] = 1 if in_trade else 0
    return pd.Series(signal, index=price.index).shift(1)


print("All 5 strategy functions loaded.")
print("  1. MA Crossover (short=20, long=50)")
print("  2. RSI (period=14, oversold=30, overbought=70)")
print("  3. Bollinger Bands (window=20, std=2.0)")
print("  4. MACD (fast=12, slow=26, signal=9)")
print("  5. Mean Reversion Z-score (window=20, threshold=1.5)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  RUN THE SCREENER
#  For each asset: run all 5 strategies on TRAIN data → pick best by Sharpe
#  Then apply the winning strategy to TEST data → see if it holds up
# ══════════════════════════════════════════════════════════════════════════════

STRATEGIES = {
    "MA Crossover":   lambda p: strategy_ma(p),
    "RSI":            lambda p: strategy_rsi(p),
    "Bollinger Bands": lambda p: strategy_bb(p),
    "MACD":           lambda p: strategy_macd(p),
    "Mean Reversion": lambda p: strategy_mr(p),
}

train_results = []   # all strategies x all assets on train data
best_strategy = {}   # asset → name of best strategy (chosen on train)
test_results  = []   # best strategy per asset applied to test data
test_dfs      = {}   # for plotting

print("=" * 70)
print("  PHASE 1 — TRAINING (2016–2020): finding best strategy per asset")
print("=" * 70)

for asset in ASSETS:
    full_price = price_data[asset]
    train_price = full_price[full_price.index < TRAIN_END]

    best_sharpe = -999
    best_name   = None
    asset_train = []

    for name, fn in STRATEGIES.items():
        pos     = fn(train_price)
        metrics = compute_metrics(train_price, pos)
        asset_train.append({
            "Asset":    asset,
            "Strategy": name,
            "Sharpe":   metrics["Sharpe"],
            "Total Ret %": metrics["Total Ret %"],
            "Max DD %": metrics["Max DD %"],
        })
        if metrics["Sharpe"] > best_sharpe:
            best_sharpe = metrics["Sharpe"]
            best_name   = name

    best_strategy[asset] = best_name
    train_results.extend(asset_train)
    print(f"  {asset}: best strategy = {best_name} (train Sharpe: {best_sharpe:.2f})")


print("\n" + "=" * 70)
print("  PHASE 2 — TESTING (2021–2026): applying chosen strategy to unseen data")
print("=" * 70)

for asset in ASSETS:
    full_price  = price_data[asset]
    test_price  = full_price[full_price.index >= TRAIN_END]
    name        = best_strategy[asset]
    fn          = STRATEGIES[name]
    pos         = fn(test_price)
    metrics     = compute_metrics(test_price, pos)

    # B&H reference
    log_ret     = np.log(test_price / test_price.shift(1))
    bah_ret     = (np.exp(log_ret.cumsum()).dropna().iloc[-1] - 1) * 100

    test_results.append({
        "Asset":           asset,
        "Best Strategy":   name,
        "OUT Sharpe":      metrics["Sharpe"],
        "OUT Strat Ret %": metrics["Total Ret %"],
        "OUT B&H Ret %":   round(bah_ret, 2),
        "OUT Max DD %":    metrics["Max DD %"],
        "Beats B&H":       metrics["Total Ret %"] > bah_ret,
    })
    test_dfs[asset] = metrics["_df"]

    print(f"  {asset} → {name}")
    print(f"    OUT Sharpe: {metrics['Sharpe']:.2f}  |  "
          f"Strat: {metrics['Total Ret %']:.1f}%  |  "
          f"B&H: {bah_ret:.1f}%  |  "
          f"Max DD: {metrics['Max DD %']:.1f}%")


# ── Final output table ─────────────────────────────────────────────────────────
results_df = pd.DataFrame(test_results).set_index("Asset")

print("\n" + "=" * 70)
print("  SCREENER RESULTS — BEST STRATEGY PER ASSET (out-of-sample)")
print("=" * 70)
print(results_df.to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  VISUALISATIONS
#  1. Summary bar chart — Sharpe per asset, colour-coded by best strategy
#  2. Equity curves — best strategy vs B&H for each asset (out-of-sample)
# ══════════════════════════════════════════════════════════════════════════════

STRATEGY_COLORS = {
    "MA Crossover":    "steelblue",
    "RSI":             "darkorange",
    "Bollinger Bands": "green",
    "MACD":            "purple",
    "Mean Reversion":  "crimson",
}

# ── Chart 1: Sharpe bar chart coloured by strategy ────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

sharpes = results_df["OUT Sharpe"]
colors  = [STRATEGY_COLORS[results_df.loc[a, "Best Strategy"]] for a in ASSETS]

bars = ax.bar(ASSETS, sharpes, color=colors, edgecolor="white", linewidth=0.5)

for bar, asset in zip(bars, ASSETS):
    strat = results_df.loc[asset, "Best Strategy"]
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f"{bar.get_height():.2f}\n{strat}",
            ha="center", va="bottom", fontsize=8)

# legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in STRATEGY_COLORS.items()]
ax.legend(handles=legend_elements, loc="upper right", fontsize=8)

ax.set_ylabel("Out-of-Sample Sharpe Ratio")
ax.set_title("Goofy Screener — Best Strategy per Asset (Out-of-Sample Sharpe)", fontsize=13)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("screener_sharpe_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: screener_sharpe_summary.png")


# ── Chart 2: Equity curves per asset (out-of-sample) ─────────────────────────
n      = len(ASSETS)
cols   = 3
rows   = (n + cols - 1) // cols
fig2, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
axes   = axes.flatten()

for i, asset in enumerate(ASSETS):
    ax    = axes[i]
    df    = test_dfs[asset]
    strat = results_df.loc[asset, "Best Strategy"]
    color = STRATEGY_COLORS[strat]

    ax.plot(df.index, df["cum_market"],   color="steelblue", linewidth=1.5,
            linestyle="--", label="Buy & Hold")
    ax.plot(df.index, df["cum_strategy"], color=color,       linewidth=1.5,
            label=strat)
    ax.axhline(y=1, color="gray", linestyle=":", linewidth=0.8)
    ax.set_title(f"{asset}  |  {strat}", fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Goofy Screener — Best Strategy Equity Curves (Out-of-Sample 2021–2026)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("screener_equity_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: screener_equity_curves.png")


# ── Chart 3: Strategy vs B&H return comparison ────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(14, 5))
x     = np.arange(len(ASSETS))
width = 0.35

strat_rets = results_df["OUT Strat Ret %"].values
bah_rets   = results_df["OUT B&H Ret %"].values

b1 = ax3.bar(x - width/2, strat_rets, width, label="Best Strategy",
             color=[STRATEGY_COLORS[results_df.loc[a, "Best Strategy"]] for a in ASSETS],
             edgecolor="white")
b2 = ax3.bar(x + width/2, bah_rets,   width, label="Buy & Hold",
             color="lightgrey", edgecolor="white")

for bar in list(b1) + list(b2):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1,
             f"{bar.get_height():.0f}%",
             ha="center", va="bottom", fontsize=8)

ax3.set_xticks(x)
ax3.set_xticklabels([
    f"{a}\n({results_df.loc[a, 'Best Strategy']})" for a in ASSETS
], fontsize=8)
ax3.set_ylabel("Total Return % (out-of-sample)")
ax3.set_title("Goofy Screener — Strategy vs Buy & Hold Return", fontsize=13)
ax3.legend()
ax3.axhline(y=0, color="black", linewidth=0.8)
ax3.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("screener_return_vs_bah.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: screener_return_vs_bah.png")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TRY YOUR OWN ASSETS
#  Change the list below and re-run from Cell 1
#  The screener will automatically find the best strategy for each one
# ══════════════════════════════════════════════════════════════════════════════

# Examples of assets you could try:
#   US stocks:        "AAPL", "MSFT", "TSLA", "AMZN", "META"
#   ASX stocks:       "WBC.AX", "ANZ.AX", "RIO.AX", "WES.AX"
#   Japanese stocks:  "9984.T" (SoftBank), "6501.T" (Hitachi)
#   ETFs:             "QQQ", "GLD", "TLT", "EEM"
#   Crypto (via YF):  "BTC-USD", "ETH-USD"

MY_ASSETS = ["AAPL", "TSLA", "MSFT", "RIO.AX", "WBC.AX"]  # <- change this

print("Downloading custom asset list...")
my_price_data = {}
for asset in MY_ASSETS:
    raw = yf.download(asset, start=TRAIN_START, end=TEST_END,
                      auto_adjust=True, progress=False)
    if raw.empty:
        print(f"  WARNING: No data for {asset} — check the ticker symbol")
    else:
        my_price_data[asset] = raw["Close"].squeeze()
        print(f"  {asset}: {len(raw)} rows — OK")

MY_ASSETS = list(my_price_data.keys())

my_results = []
my_test_dfs = {}

for asset in MY_ASSETS:
    full_price  = my_price_data[asset]
    train_price = full_price[full_price.index < TRAIN_END]
    test_price  = full_price[full_price.index >= TRAIN_END]

    # find best strategy on training data
    best_sharpe = -999
    best_name   = None
    for name, fn in STRATEGIES.items():
        pos     = fn(train_price)
        metrics = compute_metrics(train_price, pos)
        if metrics["Sharpe"] > best_sharpe:
            best_sharpe = metrics["Sharpe"]
            best_name   = name

    # apply to test data
    pos     = STRATEGIES[best_name](test_price)
    metrics = compute_metrics(test_price, pos)

    log_ret = np.log(test_price / test_price.shift(1))
    bah_ret = (np.exp(log_ret.cumsum()).dropna().iloc[-1] - 1) * 100

    my_results.append({
        "Asset":           asset,
        "Best Strategy":   best_name,
        "Train Sharpe":    round(best_sharpe, 2),
        "OUT Sharpe":      metrics["Sharpe"],
        "OUT Strat Ret %": metrics["Total Ret %"],
        "OUT B&H Ret %":   round(bah_ret, 2),
        "OUT Max DD %":    metrics["Max DD %"],
        "Beats B&H":       metrics["Total Ret %"] > bah_ret,
    })
    my_test_dfs[asset] = metrics["_df"]

my_df = pd.DataFrame(my_results).set_index("Asset")

print("\n" + "=" * 70)
print("  YOUR CUSTOM SCREENER RESULTS")
print("=" * 70)
print(my_df.to_string())

# Equity curves for custom assets
n    = len(MY_ASSETS)
cols = min(3, n)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
axes = np.array(axes).flatten() if n > 1 else [axes]

for i, asset in enumerate(MY_ASSETS):
    ax    = axes[i]
    df    = my_test_dfs[asset]
    strat = my_df.loc[asset, "Best Strategy"]
    color = STRATEGY_COLORS.get(strat, "orange")

    ax.plot(df.index, df["cum_market"],   color="steelblue", linewidth=1.5,
            linestyle="--", label="Buy & Hold")
    ax.plot(df.index, df["cum_strategy"], color=color,       linewidth=1.5,
            label=strat)
    ax.axhline(y=1, color="gray", linestyle=":", linewidth=0.8)
    ax.set_title(f"{asset}  |  {strat}", fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Custom Screener — Best Strategy Equity Curves", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("screener_custom_equity_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: screener_custom_equity_curves.png")
